[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://githubtocolab.com/Soliman2020/python_48h_notebooks/blob/master/Session_10_Classes_and_Decorators.ipynb)

# Session 10 — Classes and Decorators
## Python Fundamentals & Practical Applications — 48-Hour Edition

**STUDENT EDITION — WRITE YOUR OWN CODE**  
**Duration:** 4 classroom hours (including a 30-minute break)  
**Audience:** beginners who have completed the earlier Python sessions

> **Beginner promise:** We model familiar work items as objects, add rules as methods, and then use decorators to add reusable behavior around functions.
> **Professional habit:** Keep business rules in one place and make reusable behavior explicit.

In [ ]:
student_name = input("Enter your name: ")
print(f"{student_name}, welcome to the classes and decorators session.")

## Learning outcomes

By the end of this session, you should be able to:

1. Define a class with `__init__`.
2. Create objects and use attributes and methods.
3. Distinguish instance attributes from class attributes.
4. Validate state changes inside methods.
5. Create a readable object with `__str__`.
6. Write and apply a decorator.
7. Use `functools.wraps`, `*args`, and `**kwargs` in reusable decorators.
8. Combine classes and decorators in a small work-item system.

**Total: 240 minutes.** The retry and cache extensions are challenge work.

## 1. Classes and objects

A **class** is a blueprint. An **object** is one concrete instance created from that blueprint. A class groups related data and behavior.

In this session, a `WorkItem` has data such as `case_id`, `sector`, `status`, and `hours_open`. Its methods define allowed actions such as closing the item or checking an SLA.

- An **attribute** stores data: `item.status`.
- A **method** performs an action: `item.close()`.
- `self` refers to the current object.
- `__init__` runs when an object is created.
- `__str__` controls how an object is displayed by `print()`.

In [ ]:
class WorkItem:
    """One work item with a small set of business rules."""

    open_statuses = {"open", "in progress"}   # class attribute

    def __init__(self, case_id, sector, hours_open=0, status="open"):
        self.case_id = case_id          # instance attribute
        self.sector = sector            # instance attribute
        self.hours_open = hours_open    # instance attribute
        self.status = status            # instance attribute

    def close(self):
        self.status = "closed"
        return self.status

    def is_overdue(self, sla_hours):
        return self.hours_open > sla_hours and self.status in self.open_statuses

    def __str__(self):
        return f"{self.case_id} | {self.sector} | {self.status} | {self.hours_open} hours"


In [ ]:
# Example usage
item = WorkItem("A101", "Finance", hours_open=30)

print(item)

In [ ]:
print("Case ID:", item.case_id)

In [ ]:
print("Overdue at 24 hours:", item.is_overdue(24))

In [ ]:
item.close()

print("After closing:", item)

In [ ]:
print("Available attributes and methods:", [name for name in dir(item) if not name.startswith("__")])

### Creating the `BankAccount` Class

We define a class with the `class` keyword, then a **constructor** method called `__init__` that runs whenever we create a new object. The first parameter, `self`, is the object being created — `self.balance` means "this account's balance".

We give the constructor **defaults**: a new account starts at balance `0.0` in `EUR`, unless you say otherwise. Defaults are how you avoid forcing every caller to supply every argument.

In [ ]:
# Create a class BankAccount

class BankAccount:

    # Constructor — runs when a new account is created
    def __init__(self, account_number, holder, balance=0.0, currency='EUR'):
        self.account_number = account_number
        self.holder = holder
        self.balance = balance
        self.currency = currency

    # Method — deposit funds (rejects negative amounts)
    def deposit(self, amount):
        if amount < 0:
            raise ValueError("Deposit amount must be non-negative.")
        self.balance = self.balance + amount
        return self.balance

    # Method — withdraw funds (rejects overdraft beyond a limit)
    def withdraw(self, amount):
        if amount < 0:
            raise ValueError("Withdrawal amount must be non-negative.")
        if amount > self.balance:
            raise ValueError("Insufficient funds for this withdrawal.")
        self.balance = self.balance - amount
        return self.balance

    # Method — a readable string representation of the account
    def __str__(self):
        return f"{self.account_number} | {self.holder} | {self.balance:,.2f} {self.currency}"

#### Anatomy of the class

- `class BankAccount(object):` — declares the class. `(object)` means it inherits from Python's base object (optional in modern Python, kept here to match the original lab's style).
- `def __init__(self, ...):` — the **constructor**. The parameters after `self` are what a caller must (or may) supply when creating an account.
- `self.balance = balance` — stores the argument on the object as an **attribute**.
- `balance=0.0` and `currency='EUR'` — **default values**. If a caller omits them, these are used.
- `deposit`, `withdraw` — **methods** that change `self.balance`. Each guards against invalid input — that is the business rule living in one place, not scattered across callers.
- `__str__` — a special method Python calls when you `print()` an object, so it reads nicely instead of printing `<BankAccount object at 0x...>`.

#### Creating an Instance of `BankAccount`

Let's create Aivar Tamm's current account with an opening balance of €4,520.50.

In [ ]:
# Create an object: Aivar's current account
avivar_account = BankAccount('EE3800123456', 'Aivar Tamm', balance=4520.50)
print(avivar_account)

#### Inspect the object with `dir()`

`dir()` lists every attribute and method available on the object — including the ones Python adds by default (`__init__`, `__str__`, `__class__`, etc.). The ones **we** defined — `account_number`, `holder`, `balance`, `currency`, `deposit`, `withdraw` — appear in the list too.

In [ ]:
# List the attributes and methods available on the account
dir(avivar_account)

#### Read and change attributes

Use dot notation to read or set an attribute.

In [ ]:
# Read an attribute
avivar_account.account_number

In [ ]:
avivar_account.holder

In [ ]:
avivar_account.balance

In [ ]:
avivar_account.currency

> **Banking note:** directly assigning `account.balance = 0` is legal Python, and in the original lab that was the whole point. In real banking code you almost never let callers set `balance` directly — you force them through `deposit()` and `withdraw()` so the rules (no negative amounts, no silent overdraft, audit logging) always run. We show the direct-set below to teach the concept, but prefer the methods.

In [ ]:
# (Teaching only) directly set an attribute — bypassing the methods
avivar_account.balance = 1.0
avivar_account.balance

#### Call methods

Now use the real interface — `deposit()` and `withdraw()` — and watch the balance change. Each method **returns** the new balance, so you can chain or log it.

In [ ]:
# Deposit, then withdraw, printing the balance each time
print('Balance before:        ', avivar_account.balance)
avivar_account.deposit(500)
print('After deposit(500):     ', avivar_account.balance)
avivar_account.withdraw(120)
print('After withdraw(120):    ', avivar_account.balance)

#### Use the defaults

If we only supply the account number and holder, the constructor fills in the defaults: balance `0.0`, currency `EUR`. This mirrors opening a brand-new account with no initial deposit.

In [ ]:
# Open a new account relying on the default balance and currency
new_account = BankAccount('EE3800999777', 'Liis Rebane',balance=6_000)
print(new_account)
print('Balance:', new_account.balance, '| Currency:', new_account.currency)

#### The business rules actually fire

Try a withdrawal that exceeds the balance, or a negative deposit. The methods raise a `ValueError` — the kind of guard that prevents a teller script from silently pushing an account millions into the red.

In [ ]:
# The method enforces the rule — this raises ValueError
try:
    new_account.withdraw(5_000)
except ValueError as e:
    print("Blocked:", e)

print('Balance unchanged:', new_account.balance)

### Guided activity 1 — build a `WorkItem` class

Create a class with `case_id`, `priority`, `status`, and `hours_open`. Add methods that:

1. Mark the item as `closed`.
2. Return `True` when the item exceeds a supplied SLA.
3. Display a readable summary.

Use one object to test each method.

In [ ]:
# TODO: define your class and test it here





## 2. Attributes, class attributes, and validation

An instance attribute belongs to one object. A class attribute is shared as a default by all objects.

Validation belongs inside the method that changes state. This prevents every caller from having to remember the same rule.

In [ ]:
class Queue:
    default_limit = 3  # class attribute shared by queue objects

    def __init__(self, name, limit=None):
        self.name = name
        self.limit = limit if limit is not None else Queue.default_limit
        self.items = []

    def add(self, item):
        if len(self.items) >= self.limit:
            raise ValueError("Queue is full.")
        self.items.append(item)
        return len(self.items)

    def open_items(self):
        return [item for item in self.items if item.status in WorkItem.open_statuses]


### Guided activity 2 — add a business rule

Extend `WorkItem` or `Queue` with one validation rule. For example:

- Reject a negative `hours_open` value.
- Prevent a closed item from being closed again.
- Prevent a queue from exceeding its limit.

Test both a valid operation and an invalid operation with `try`/`except`.

In [ ]:
# TODO: add and test one validation rule





## 3. Decorators

A decorator is a function that receives another function and returns a new function with extra behavior. The `@decorator` syntax applies it when the function is defined.

A decorator should usually:

1. Accept the original function.
2. Define a wrapper.
3. Run behavior before or after the original function.
4. Return the original result.
5. Use `@wraps` so the original function name and documentation are preserved.

In [ ]:
from functools import wraps

def announce(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        print(f"Starting {func.__name__}")
        result = func(*args, **kwargs)
        print(f"Finished {func.__name__}")
        return result

    return wrapper

@announce
def make_label(case_id, status="open"):
    return f"{case_id}: {status}"

print(make_label("A101", status="closed"))
print("Function name:", make_label.__name__)

### Guided activity 3 — write a decorator

Write a `log_call` decorator that prints the function name and its arguments before calling the function. It must:

- work with positional and keyword arguments;
- return the original result;
- preserve the function name with `@wraps`.

Apply it to a small function such as `calculate_sla(hours_open, sla_hours)`.

In [ ]:
# TODO: write and apply log_call





## 4. Practical decorators

Decorators are useful for cross-cutting behavior such as validation, timing, logging, and caching. The decorated function still performs its main job; the decorator adds a reusable rule around it.

In [ ]:
import time

def timer(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        elapsed = time.perf_counter() - start
        print(f"{func.__name__} took {elapsed:.6f} seconds")
        return result

    return wrapper

def require_positive(func):
    @wraps(func)
    def wrapper(amount, *args, **kwargs):
        if amount <= 0:
            raise ValueError("Amount must be positive.")
        return func(amount, *args, **kwargs)

    return wrapper

@timer
@require_positive
def sum_squares(amount):
    return sum(number ** 2 for number in range(amount))

print(sum_squares(1000))


In [ ]:
try:
    sum_squares(0)
except ValueError as error:
    print("Blocked:", error)

### Guided activity 4 — decorators on methods

Apply a decorator to a class method. Remember that a method receives `self` first. Create a decorator that prevents a work item from being updated with a negative number of hours. Test a valid and invalid update.

In [ ]:
# TODO: create a method decorator and apply it to a class method





## Practice tasks

Use comments to document each design decision and leave a small test after each non-trivial method or decorator.

### Applied task A — `S10-E01`
Create a `WorkItem` class with four attributes, a `close()` method, an `is_overdue()` method, and a readable `__str__()` method.

In [ ]:
# Your code here





### Applied task B — `S10-E02`
Create a `WorkQueue` class that stores work items, rejects items after a maximum capacity, and returns only open items.

In [ ]:
# Your code here





### Applied task C — `S10-E03`
Write an `announce` or `log_call` decorator. Apply it to two functions with different arguments and confirm that both return their normal results.

In [ ]:
# Your code here





### Applied task D — `S10-E04`
Write a decorator that rejects non-positive numeric input. Apply it to a function that calculates an SLA percentage or a total amount.

In [ ]:
# Your code here





### Challenge — `S10-E05`
Use two decorators on one function: one for timing and one for logging. Explain which decorator runs first and why `@wraps` matters.

In [ ]:
# Your code here





## Guided mini-project — decorated work-item queue

Build a small work-item processing system. Your solution must:

1. Define a `WorkItem` class.
2. Define a `WorkQueue` class that adds and closes items.
3. Validate at least one state-changing operation.
4. Add a logging decorator to one queue method.
5. Add a timing decorator to one function or method.
6. Produce a summary of open, closed, and overdue items.
7. Include at least one `assert` self-check.

Keep the classes small. The goal is to demonstrate object state and reusable decorators, not to build a production queue.

In [ ]:
# TODO: complete the mini-project





## Optional extension

Choose one:

- Add a simple cache decorator for a function with hashable arguments.
- Add a retry decorator for a function that may raise a temporary exception.
- Add a decorator that records each state change in a list.
- Add a class attribute that records the number of objects created.

Explain one limitation of your extension.

In [ ]:
# Optional: extend the classes or decorators





## Exit check — answer from memory

1. What is a class?
2. What is an object?
3. What does `self` refer to?
4. What is the purpose of `__init__`?
5. What is the difference between an instance attribute and a class attribute?
6. What does a decorator receive and return?
7. Why is `functools.wraps` useful?
8. Why do reusable decorators often use `*args` and `**kwargs`?
9. What happens when multiple decorators are applied to one function?

## Common mistakes and next steps

- Remember `self` in instance methods.
- Use instance attributes for per-object state.
- Keep validation beside the state-changing method.
- Do not forget to return the original function's result from a decorator.
- Use `@wraps` when writing reusable decorators.
- Use `*args` and `**kwargs` when the decorator should support different function signatures.
- Test invalid inputs as well as successful inputs.

**Next step:** use classes to model larger systems and decorators to add consistent logging, validation, timing, or caching behavior.